**Note**: To proceed, please ensure the Notebook Service is started. This is the same service created in Notebook #1.

# Part 2: Openflow Gen2 (SOM) — Runtime & Connector Setup

This notebook creates the Openflow runtime and deploys the PostgreSQL CDC connector to start ingesting data from Snowflake Managed Postgres into the `RETAIL_HOL_ICEBERG` database.

## What we'll create

| Object | Name | Purpose |
|--------|------|---------|
| Runtime | `POSTGRES_RUNTIME_SOM` | Gen2 runtime for CDC ingestion |
| Tasks | Suspend/Resume runtime on schedule | Cost control automation |
| Connector | `POSTGRES_INGEST` | PostgreSQL CDC (Guided Wizard) connector |

## Prerequisites (pre-provisioned)

- Everything from **Notebook 00** (roles, deployment `SPCS_SOM` must be **ACTIVE**)
- Role: `RETAIL_HOL_ROLE`
- Warehouse: `RETAIL_HOL_WH`
- Databases: `RETAIL_HOL_ADMIN`, `RETAIL_HOL_ICEBERG`
- Network Policy: `RETAIL_HOL_NETWORK_POLICY`
- EAI: `RETAIL_HOL_POSTGRES_EAI`
- Secret: `RETAIL_HOL_ADMIN.SECRETS.RETAIL_HOL_PG_PASSWORD`

## Step 0: Grant Secret Access to Runtime Service Role

Now that the Postgres password secret has been created in Notebook 01, grant the runtime service role permission to read it. This is required for the connector to authenticate with the Postgres instance.

In [ ]:
%%sql -r dataframe_prereq
USE ROLE ACCOUNTADMIN;

-- Read the Postgres password secret
GRANT READ ON SECRET RETAIL_HOL_ADMIN.SECRETS.RETAIL_HOL_PG_PASSWORD TO ROLE POSTGRES_INGEST_SERVICE_ROLE;

### ⚠️ Check Openflow Deployment ACTIVE Status
Run the cell below and check the `state` column — proceed only when it shows **ACTIVE**. The deployment typically takes ~15 minutes from the time you ran the CREATE command in Notebook 00.

In [ ]:
%%sql -r dataframe_3
USE ROLE OPENFLOW_ADMIN;
SHOW OPENFLOW DEPLOYMENTS;

## Step 1: Create the Gen2 Runtime

The **runtime** is the compute environment that runs your connectors. Key settings:
- `NODE_TYPE = MEDIUM` — minimum requirement for CDC workloads
- `EXECUTE_AS_ROLE` — the service role that the connector uses to access Snowflake objects
- `EXTERNAL_ACCESS_INTEGRATIONS` — enables outbound network access to your Postgres instance

> **Note:** The runtime takes 2-3 minutes to reach ACTIVE state after creation.

In [ ]:
%%sql -r dataframe_6
USE ROLE OPENFLOW_ADMIN;
USE DATABASE OPENFLOW_DB;
USE SCHEMA OPENFLOW_SCHEMA;

CREATE OPENFLOW RUNTIME IF NOT EXISTS OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_RUNTIME_SOM
  IN DEPLOYMENT SPCS_SOM
  NODE_TYPE = MEDIUM
  MIN_NODES = 1
  MAX_NODES = 1
  EXECUTE_AS_ROLE = POSTGRES_INGEST_SERVICE_ROLE
  EXTERNAL_ACCESS_INTEGRATIONS = (RETAIL_HOL_POSTGRES_EAI)
  DISPLAY_NAME = 'POSTGRES_RUNTIME_SOM'
  COMMENT = "SOM RUNTIME FOR INGESTING FROM SNOWFLAKE POSTGRES";

### ⚠️ Check Runtime Status

> **Wait here until the `state` column shows `ACTIVE`.** Runtime creation typically takes **~5 minutes**. Re-run the cell below until you see `ACTIVE` before proceeding to Step 2.

In [ ]:
%%sql -r dataframe_7
-- Verify the runtime was created and check its status
DESCRIBE OPENFLOW RUNTIME POSTGRES_RUNTIME_SOM;

### Cost Control: Scheduled Suspend/Resume

These tasks automatically suspend the runtime at 5 PM and resume it at 6 AM (Pacific), preventing unnecessary overnight compute charges. **Note**: This is a best practice for **non-production** environments.

In [ ]:
%%sql -r dataframe_8
USE ROLE ACCOUNTADMIN;

-- Suspend runtime every day at 5 PM Pacific
CREATE TASK IF NOT EXISTS OPENFLOW_DB.OPENFLOW_SCHEMA.SUSPEND_POSTGRES_RUNTIME_SOM
  WAREHOUSE = RETAIL_HOL_WH
  SCHEDULE = 'USING CRON 0 17 * * * America/Los_Angeles'
AS
  ALTER OPENFLOW RUNTIME OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_RUNTIME_SOM SUSPEND;

-- Resume runtime every day at 6 AM Pacific
CREATE TASK IF NOT EXISTS OPENFLOW_DB.OPENFLOW_SCHEMA.RESUME_POSTGRES_RUNTIME_SOM
  WAREHOUSE = RETAIL_HOL_WH
  SCHEDULE = 'USING CRON 0 6 * * * America/Los_Angeles'
AS
  ALTER OPENFLOW RUNTIME OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_RUNTIME_SOM RESUME;

## Step 2: Deploy the PostgreSQL CDC Gen2 Connector (Snowsight UI)

> **New in Openflow Gen2:** The **Guided Wizard** is a brand-new, fully UI-driven experience for deploying connectors. Unlike the classic NiFi canvas approach, there's no drag-and-drop flow configuration required — the wizard walks you through each setting step by step, making it faster and more accessible.

Once the runtime reaches **ACTIVE** status, deploy the connector using the guided wizard:

1. In Snowsight, navigate to **Ingestion → Openflow**.
2. On the Openflow page, click on **Launch Openflow** button located at the the top-right corner.
3. To access the required permissions, please navigate to the top-right corner of the page, locate your username, and switch your role to **OPENFLOW_ADMIN**.
4. Go to the **Runtime** tab and click on your newly created runtime. Authenticate/authorize if prompted (allow the Openflow Dataplane to access your Snowflake account under your active role).
5. On the **Overview** tab, click **"View more connectors"**.
6. Under the **Gen2** group, find **"PostgreSQL CDC (Guided Wizard)"** and click **Install**.
   > ⚠️ Do **not** install the plain "PostgreSQL" connector — it must be the **Guided Wizard** version.
7. Select the `OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_RUNTIME_SOM` runtime, name your connector `POSTGRES_INGEST`, and click **Begin Installation**.
8. Follow the guided wizard to configure source connection, destination, and tables to replicate. All configuration details for each wizard step are provided in **Step 3** of this notebook.
9. Authenticate/authorize if prompted (**Allow** the Openflow Dataplane to access your Snowflake account under your active role).

> **Tip:** Users wanting to deploy a large fleet of connectors with similar configuration can use the SQL API combined with a templating engine like Jinja2. Use the guided wizard for your first connector, then generalize it for automation.

## Step 3: Configure the PostgreSQL CDC Connector (Guided Wizard)

After the connector installation begins (Step 2), the guided wizard walks you through 7 configuration screens.


---

### Wizard Step 1: Source

| Field | Value | Notes |
|-------|-------|-------|
| Source Database Connection URL | `jdbc:postgresql://<HOST>:5432/postgres` | Get `<HOST>` from `DESCRIBE POSTGRES INSTANCE RETAIL_HOL_PG` |
| Source Database JDBC Driver | Upload `.jar` from [jdbc.postgresql.org](https://jdbc.postgresql.org/download/) | Download the latest version of driver from the website just by pasting the link in a new browser (exp. postgresql-42.7.13.jar). Check **Reference asset** after uploading |
| Source Database User | `snowflake_admin` | Default admin user on Snowflake Managed Postgres |
| Source Database Password | Select secret from dropdown | `RETAIL_HOL_ADMIN.SECRETS.RETAIL_HOL_PG_PASSWORD` |
| Source Database Publication Name | `openflow_retail_pub` | Created in Notebook 01 |

---

### Wizard Step 2: Replication Table Schema

- Choose **Manual** selection
- Check **only** the `bronze` schema (all 5 tables should be selected)
- **Uncheck** the `public` schema


---

### Wizard Step 3: Replication Columns

- Leave **all columns selected** (default)
- Leave **"Always include newly added columns"** toggle on  — this ensures any columns added to the source tables in the future are automatically captured

---

### Wizard Step 4: Destination Details

| Field | Value | Notes |
|-------|-------|-------|
| Snowflake Destination Database | `RETAIL_HOL_ICEBERG` | Must already exist |
| Snowflake Warehouse | `RETAIL_HOL_WH` | Used for merge operations |
| Destination Schema Strategy | `{schema}` | Preserves source schema name as destination schema |
| Object Identifier Resolution | `Case Insensitive` | Default; do not change after initial start |
| Oversized Value Strategy | `Set Null` | Nulls values exceeding Snowpipe Streaming max size |
| Table Storage Format | `Iceberg` | Creates Iceberg tables instead of regular tables |
| Iceberg Version | `Version 3` | Direct ingestion to Iceberg is a Preview feature in the connector; cannot be changed after ingestion starts |


---

### Wizard Step 5: Tuning (No change needed. Go with defaults.)

| Field | Value | Notes |
|-------|-------|-------|
| Merge Task Schedule CRON | `* * * * * ?` | Continuous merge (every minute). Use a less frequent schedule for cost optimization in production. |
| Concurrent Snapshot Queries | `2` | Number of parallel snapshot queries to the source. Increase for more tables but higher source load. |


---

### Wizard Step 6: Migration (No change needed. Go with defaults.)

| Field | Value | Notes |
|-------|-------|-------|
| Ingestion Type | `Full` | Performs initial snapshot + ongoing CDC. Choose `Incremental` to skip snapshot. |
| Replication Slot Name | *(leave blank)* | Connector auto-creates a unique replication slot. |

---

### Wizard Step 7: Summary

The final page displays **Review configuration** — a consolidated view of all your settings grouped by Source, Data replication, and Destination details.

1. Review all values to confirm they are correct
2. Click **Verify configuration** to run validation
3. Once validated, click **Apply and Continue** to exit the wizard

> **Important:** After completing the wizard, the connector will be in a **Stopped** state. You must start it manually:
> - In the Openflow UI, click the **three-dot menu (⋮)** on the connector and select **Start**
> - Or via SQL: `ALTER OPENFLOW CONNECTOR OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_INGEST START;`

Once started, the connector will:
1. Perform an initial full snapshot of the 5 tables in the `bronze` schema
2. Switch to CDC mode and continuously replicate changes from the Postgres WAL

Note: If you don't see data in your destination tables please go check the observability dashboard (explained in Step 4). 

## Step 4 (Optional): Operational Best Practices: Monitoring, Alerting & Upgrades

Once your connector is running, use these tools to maintain visibility and reliability.

---

### Connector Observability Dashboard

In Snowsight, navigate to **Ingestion → Openflow → Connector Observability** → In the dropdown for event table, select the event table **"OPENFLOW_DB.OPENFLOW_SCHEMA.EVENTS"** — the built-in observability dashboard shows:
- Connector status across all runtimes (Running connectors)
- Throughput metrics (records/sec, bytes/sec)
- Replication lag and latency
- Table-level sync status and error counts

---

### Querying the Event Table

The event table captures runtime logs, errors, and telemetry. Use it for troubleshooting and auditing:

```sql
-- Recent errors from the Openflow runtime
SELECT TIMESTAMP, RESOURCE_ATTRIBUTES, RECORD_ATTRIBUTES, VALUE
FROM OPENFLOW_DB.OPENFLOW_SCHEMA.EVENTS
WHERE RECORD_ATTRIBUTES:"severity_text" = 'ERROR'
ORDER BY TIMESTAMP DESC
LIMIT 20;

-- All events in the last hour
SELECT *
FROM OPENFLOW_DB.OPENFLOW_SCHEMA.EVENTS
WHERE TIMESTAMP > DATEADD('hour', -1, CURRENT_TIMESTAMP())
ORDER BY TIMESTAMP DESC;
```

---

### Setting Up Alerts with Openflow Templates

Snowflake provides **pre-built alert templates** specifically for Openflow monitoring. These templates cover common failure scenarios out of the box:

To set up Openflow alerts:
1. In Snowsight, navigate to **Monitoring → Alerts**
2. Click **+ Create Alert**
3. Select from the **Openflow** template category — available templates include:
   - Connector failure / stopped unexpectedly
   - Replication lag exceeds threshold
   - Runtime unhealthy or unresponsive
4. Configure the notification channel (email, Slack, webhook) and thresholds. This will require a notification integration. 
5. Activate the alert

> **Tip:** Start with the connector failure and replication lag templates — these catch the most common production issues early.

Alternatively, you can create custom alerts using the event table for more specific conditions:

```sql
CREATE OR REPLACE ALERT OPENFLOW_DB.OPENFLOW_SCHEMA.OPENFLOW_ERROR_ALERT
  WAREHOUSE = RETAIL_HOL_WH
  SCHEDULE = '5 MINUTE'
  IF (EXISTS (
    SELECT 1
    FROM OPENFLOW_DB.OPENFLOW_SCHEMA.EVENTS
    WHERE RECORD_ATTRIBUTES:"severity_text" = 'ERROR'
      AND TIMESTAMP > DATEADD('minute', -5, CURRENT_TIMESTAMP())
  ))
  THEN
    CALL SYSTEM$SEND_EMAIL(
      'my_email_integration',
      'your-email@company.com',
      'Openflow Connector Error Detected',
      'One or more errors occurred in the Openflow runtime in the last 5 minutes. Check the event table for details.'
    );

ALTER ALERT OPENFLOW_DB.OPENFLOW_SCHEMA.OPENFLOW_ERROR_ALERT RESUME;
```

---

### Runtime Upgrades

Snowflake periodically releases new runtime versions with performance improvements and bug fixes. Upgrade with:

```sql
-- Check if an upgrade is available
SHOW OPENFLOW RUNTIMES;

-- Apply the upgrade (runtime restarts automatically)
ALTER OPENFLOW RUNTIME OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_RUNTIME_SOM UPGRADE;
```

> **Tip:** Schedule upgrades during low-traffic windows. The runtime briefly restarts during an upgrade, pausing ingestion momentarily.

## Step 5: Clean Up (DO NOT RUN - This is a representation of the API capabilities in OpenFlow Gen2)

To terminate the Openflow components, the following commands must be executed in the specified order. Each step must complete before the next.

> ⚠️ **Do not run these during the lab** — they will destroy your Openflow infrastructure.

In [ ]:
%%sql -r dataframe_9
-- ⚠️ CLEANUP ONLY — do not run during the lab

-- Drop Connector
ALTER OPENFLOW CONNECTOR OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_INGEST STOP;
ALTER OPENFLOW CONNECTOR OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_INGEST TERMINATE;
DROP OPENFLOW CONNECTOR OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_INGEST;

-- Drop Runtime
ALTER OPENFLOW RUNTIME OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_RUNTIME_SOM SUSPEND;
ALTER OPENFLOW RUNTIME OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_RUNTIME_SOM TERMINATE;
DROP OPENFLOW RUNTIME OPENFLOW_DB.OPENFLOW_SCHEMA.POSTGRES_RUNTIME_SOM;

-- Drop Deployment
ALTER OPENFLOW DEPLOYMENT SPCS_SOM TERMINATE;
DROP OPENFLOW DEPLOYMENT IF EXISTS SPCS_SOM;

## Recap: What We Built

In this notebook, we created the Openflow Gen2 runtime and configured the PostgreSQL CDC connector to replicate data from Snowflake Managed Postgres into Iceberg tables.

### Infrastructure provisioned

| Object | Name | Purpose |
|--------|------|---------|
| Runtime | `POSTGRES_RUNTIME_SOM` | Gen2 CDC engine running inside the `SPCS_SOM` deployment |
| Tasks | Suspend @ 5 PM / Resume @ 6 AM | Automated cost control for non-production hours |
| Connector | `POSTGRES_INGEST` | PostgreSQL CDC (Guided Wizard) connector |

### Key concepts covered

- **Openflow Gen2 Runtime** — a lightweight compute engine that runs connectors inside an Openflow deployment; `EXECUTE_AS_ROLE` determines the identity used to access Snowflake objects
- **Guided Wizard connector deployment** — a new, fully UI-driven experience that replaces the classic NiFi canvas; walks through source, replication, destination, tuning, and migration in 7 steps
- **Iceberg as destination** — the connector writes directly to Iceberg V3 tables in `RETAIL_HOL_ICEBERG.BRONZE`, enabling open-format storage from day one
- **Full ingestion mode** — the connector first snapshots all existing rows, then switches to continuous CDC from the Postgres WAL
- **Cost control** — scheduled tasks suspend the runtime at 5 PM and resume at 6 AM Pacific to avoid overnight charges in non-production environments

### Next Steps

In notebook `03-retail-hol-transformations-managed-iceberg.ipynb`, we'll take the raw CDC data in the `bronze` schema and build Silver and Gold layers using Dynamic Iceberg Tables — applying transformations, aggregations, and business logic to make the data analytics-ready.